# Análise Exploratória de Dados — Internações Hospitalares

Começo entendendo a estrutura do dataset antes de qualquer limpeza. Como expliquei no `data/README.md`, esses dados são simulados — o SIH real do DATASUS não sai em CSV pronto para download, então construí um gerador que reproduz a mesma estrutura de campos e relações estatísticas plausíveis entre idade, especialidade, permanência e custo. O pipeline que vou construir aqui funciona do mesmo jeito quando eu trocar pelos dados reais, porque as colunas seguem o mesmo layout.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

## 1. Carregando os dados

O dataset tem duas tabelas: as internações propriamente ditas e o cadastro de hospitais com a capacidade de leitos. Vou carregar as duas e olhar o formato básico antes de qualquer outra coisa.

In [ ]:
internacoes = pd.read_csv('../data/raw/internacoes_simuladas.csv', parse_dates=['data_internacao', 'data_saida'])
hospitais = pd.read_csv('../data/raw/hospitais_simulados.csv')

print(f'Internações: {internacoes.shape[0]:,} linhas, {internacoes.shape[1]} colunas')
print(f'Hospitais: {hospitais.shape[0]} unidades cadastradas')
internacoes.head()

## 2. Entendendo cada variável

Antes de limpar qualquer coisa, preciso saber o que cada coluna representa na prática hospitalar: idade e sexo do paciente, especialidade e diagnóstico responsáveis pela internação, datas de entrada e saída, valor da internação (equivalente à AIH do SUS), e os dois desfechos que mais importam para o projeto — óbito e readmissão em 30 dias.

In [ ]:
print('Tipos de dados:')
print(internacoes.dtypes)
print()
print('Valores nulos por coluna:')
print(internacoes.isnull().sum())

## 3. Duplicatas e consistência das datas

Verifico duplicatas exatas e também se alguma internação tem data de saída anterior à data de entrada — isso seria um erro de registro que precisaria ser tratado antes de qualquer análise.

In [ ]:
duplicatas = internacoes.duplicated(subset='internacao_id').sum()
datas_invalidas = (internacoes['data_saida'] < internacoes['data_internacao']).sum()

print(f'Internações duplicadas: {duplicatas}')
print(f'Datas inconsistentes (saída antes da entrada): {datas_invalidas}')

## 4. Estatísticas descritivas

Idade, tempo de permanência e valor da internação são as três variáveis numéricas centrais do projeto. Quero ver a distribuição de cada uma antes de decidir se preciso tratar outliers.

In [ ]:
internacoes[['idade', 'dias_permanencia', 'valor_total']].describe()

## 5. Distribuições

Tempo de permanência e valor da internação devem ter cauda longa à direita — a maioria dos casos é rápida e barata, mas os poucos casos graves (UTI, permanência prolongada) esticam bastante a distribuição. Vale visualizar em vez de olhar só a média.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(internacoes['dias_permanencia'], bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Tempo de permanência (dias)')
sns.histplot(internacoes['idade'], bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('Idade dos pacientes')
sns.histplot(internacoes['valor_total'], bins=30, ax=axes[2], color='steelblue')
axes[2].set_title('Valor da internação (R$)')
plt.tight_layout()
plt.savefig('../reports/figures/distribuicoes.png', dpi=150)
plt.show()

print(f"Permanência — média: {internacoes['dias_permanencia'].mean():.1f} dias | mediana: {internacoes['dias_permanencia'].median():.0f} dias")

## 6. Sazonalidade — internações totais vs respiratórias

Um dos indicadores do projeto é a demanda por leitos ao longo do tempo. Antes de modelar isso, quero confirmar visualmente que existe sazonalidade real — em especial nas internações respiratórias, que no Brasil tendem a concentrar no inverno (junho a agosto).

In [ ]:
internacoes['mes'] = internacoes['data_internacao'].dt.month
respiratorias_mensal = internacoes[
    internacoes['cid_capitulo'] == 'Doenças do aparelho respiratório'
].groupby('mes').size()

fig, ax = plt.subplots()
internacoes.groupby('mes').size().plot(kind='bar', ax=ax, color='steelblue', alpha=0.5, label='Todas as internações')
respiratorias_mensal.plot(kind='line', ax=ax, color='crimson', marker='o', label='Respiratórias')
ax.set_title('Sazonalidade mensal — internações totais vs respiratórias')
ax.set_xlabel('Mês')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/sazonalidade_mensal.png', dpi=150)
plt.show()

## 7. Correlações entre variáveis numéricas

Quero entender a relação entre idade, tempo de permanência, valor da internação, óbito e readmissão — isso ajuda a decidir quais variáveis levar para o classificador de risco no notebook 04.

In [ ]:
cols_numericas = ['idade', 'dias_permanencia', 'valor_total', 'obito', 'readmissao_30d']
corr = internacoes[cols_numericas].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlação entre variáveis')
plt.tight_layout()
plt.savefig('../reports/figures/correlacao.png', dpi=150)
plt.show()

## 8. Especialidades e capacidade instalada

Por fim, olho a distribuição de internações por especialidade e comparo a capacidade de leitos entre os tipos de gestão hospitalar — isso já dá uma primeira pista de quais hospitais podem estar sob mais pressão.

In [ ]:
print('Internações por especialidade:')
print(internacoes['especialidade'].value_counts())
print()
print('Leitos totais por tipo de gestão:')
print(hospitais.groupby('tipo_gestao')['leitos_totais'].describe()[['count', 'mean', 'min', 'max']])

## 9. Conclusões da EDA

Pontos que levo para a próxima etapa:

- Não há duplicatas nem datas inconsistentes — o dataset já nasce limpo nesse sentido, mas ainda vou aplicar as validações de sanidade em `load_data.py` como boa prática, porque isso deixa o pipeline preparado para o dia em que eu trocar pelos dados reais do SIH.
- Tempo de permanência e valor da internação têm cauda longa — confirma que UTI e casos graves puxam a média para cima. Vou usar a mediana como referência complementar no relatório final.
- Existe sazonalidade real nas internações respiratórias, concentrada no meio do ano — isso justifica usar um modelo de séries temporais com componente sazonal (Prophet) em vez de uma média móvel simples para prever demanda por leitos.
- A correlação entre idade e readmissão é a mais forte entre as variáveis numéricas isoladas, mas ainda fraca (abaixo de 0.15) — isso já sinaliza que o classificador de risco não vai ter uma variável "mágica" sozinha, e que o desempenho dele vai depender de combinar várias features fracas.

No próximo notebook, aplico a limpeza formal, calculo a ocupação diária de leitos e monto as bases que alimentam os três modelos do projeto.